In [1]:
print("hello")

hello


In [2]:


pdf_data_path = r'C:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\data_from_s3\SupplyCode.pdf'

ARTIFACTS_DIR = 'artifacts'
PDF_TEXT_CLEAN_PATH = 'pdf_text_clean_path'

In [3]:
from dataclasses import dataclass

@dataclass
class ArtifactDir:
    pdf_cleaned_data: str


In [6]:
import os



class ArtifactConfig:
    def __init__(self, pdf_data_path: str) -> None:
        self.pdf_data_path = pdf_data_path

        # Fix: os.path.join with single arg is a no-op — build from cwd explicitly
        self.artifacts_dir = os.path.join(os.getcwd(), ARTIFACTS_DIR)
        os.makedirs(self.artifacts_dir, exist_ok=True)

        # Fix: added os.makedirs for the subfolder too
        self.pdf_text_clean_path = os.path.join(
            self.artifacts_dir,
            PDF_TEXT_CLEAN_PATH
        )
        os.makedirs(self.pdf_text_clean_path, exist_ok=True)

In [8]:
import os
from pathlib import Path

import re
from typing import Dict, Any
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document


class PDFCleaner:
    """
    Production-grade PDF text cleaner for RAG pipelines
    """

    def __init__(self,
                 config: ArtifactConfig = None,
                 remove_headers=True,
                 normalize_text=True,
                 remove_special_chars=False,
                 deduplicate=True):

        self.config = config
        self.remove_headers = remove_headers
        self.normalize_text = normalize_text
        self.remove_special_chars = remove_special_chars
        self.deduplicate = deduplicate

    # -----------------------------
    # Core Cleaning Pipeline
    # -----------------------------
    def clean(self, text: str, metadata: Dict[str, Any] = None) -> Dict[str, Any]:
        """
        Main pipeline function

        Args:
            text: Raw extracted PDF text
            metadata: Optional metadata (page, source, etc.)

        Returns:
            Dict with cleaned text + metadata
        """

        if not text:
            return {"text": "", "metadata": metadata or {}}

        text = self._remove_page_numbers(text)

        if self.remove_headers:
            text = self._remove_headers_footers(text)

        text = self._fix_broken_words(text)
        text = self._fix_line_breaks(text)

        if self.normalize_text:
            text = self._normalize_text(text)

        if self.remove_special_chars:
            text = self._remove_special_characters(text)

        if self.deduplicate:
            text = self._deduplicate_lines(text)

        return {
            "text": text.strip(),
            "metadata": metadata or {}
        }

    # -----------------------------
    # Cleaning Steps
    # -----------------------------

    def _remove_page_numbers(self, text: str) -> str:
        return re.sub(r'Page\s*\d+\s*(of\s*\d+)?', '', text, flags=re.IGNORECASE)

    def _remove_headers_footers(self, text: str) -> str:
        lines = text.split("\n")
        if len(lines) < 5:
            return text
        return "\n".join(lines[1:-1])

    def _fix_broken_words(self, text: str) -> str:
        return re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)

    def _fix_line_breaks(self, text: str) -> str:
        return re.sub(r'\n+', ' ', text)

    def _normalize_text(self, text: str) -> str:
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def _remove_special_characters(self, text: str) -> str:
        return re.sub(r'[^\w\s.,%-]', '', text)

    def _deduplicate_lines(self, text: str) -> str:
        seen = set()
        unique_lines = []
        for line in text.split('. '):
            line = line.strip()
            if line and line not in seen:
                seen.add(line)
                unique_lines.append(line)
        return '. '.join(unique_lines)


def extract_text_from_pdf(pdf_data_path: str) -> str:
    loader = PyPDFLoader(pdf_data_path)
    pages = loader.load()
    text = ""
    for page in pages:
        text += page.page_content + "\n"
    return text


if __name__ == "__main__":
    # Initialize config and cleaner
    config = ArtifactConfig(pdf_data_path=pdf_data_path)
    cleaner = PDFCleaner(config=config)

    print(f"Extracting text from: {pdf_data_path}")
    raw_text = extract_text_from_pdf(pdf_data_path=pdf_data_path)

    print("Cleaning text...")
    cleaned_result = cleaner.clean(raw_text)

    # Fix: cleaned_text was never saved — write output to the configured path
    output_file = os.path.join(config.pdf_text_clean_path, "cleaned_text.txt")
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(cleaned_result["text"])
    print(f"Cleaned text saved to: {output_file}")

    artifacts_dir_pdf = ArtifactDir(
        pdf_cleaned_data=config.pdf_text_clean_path
    )
    print(f"ArtifactDir: {artifacts_dir_pdf}")

# python -m rag_project.data_processing.text_clean

Extracting text from: C:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\data_from_s3\SupplyCode.pdf
Cleaning text...
Cleaned text saved to: c:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\notebooks\artifacts\pdf_text_clean_path\cleaned_text.txt
ArtifactDir: ArtifactDir(pdf_cleaned_data='c:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\notebooks\\artifacts\\pdf_text_clean_path')


In [11]:
print(type(cleaned_result))

<class 'dict'>


In [12]:
output_file

'c:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\notebooks\\artifacts\\pdf_text_clean_path\\cleaned_text.txt'

In [13]:
artifacts_dir_pdf

ArtifactDir(pdf_cleaned_data='c:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\notebooks\\artifacts\\pdf_text_clean_path')